# Data cleaning & Preprocessing

> TODO adicionar descrição breve sobre o que fiz

## Data cleaning

In [197]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import missingno as msno
sns.set_theme(style='darkgrid')
import matplotlib.pyplot as plt

In [198]:
data = pd.read_excel('Assignment/DataSets/UCMF.xls')

data = data.rename(columns={
    'IDADE': 'Idade',
    'PULSOS':'Pulsos',
    'PA SISTOLICA': 'SBP',
    'PA DIASTOLICA': 'DBP',
    'PPA': 'Result SBP-DBP',
    'NORMAL X ANORMAL': 'Patologia',
    'SOPRO' : 'Sopro',
    'SEXO':'Sexo',
    'HDA 1': 'HDA1',
    'MOTIVO1' : 'Motivo1',
    'MOTIVO2': 'Motivo2'
})

In [199]:
data.columns

Index(['ID', 'Peso', 'Altura', 'IMC', 'Atendimento', 'DN', 'Idade', 'Convenio',
       'Pulsos', 'SBP', 'DBP', 'Result SBP-DBP', 'Patologia', 'B2', 'Sopro',
       'FC', 'HDA1', 'HDA2', 'Sexo', 'Motivo1', 'Motivo2'],
      dtype='object')

### Remove Irrelevant features

In [200]:
data = data.drop(['ID','Atendimento','DN','Convenio'],axis=1)

In [201]:
data.columns

Index(['Peso', 'Altura', 'IMC', 'Idade', 'Pulsos', 'SBP', 'DBP',
       'Result SBP-DBP', 'Patologia', 'B2', 'Sopro', 'FC', 'HDA1', 'HDA2',
       'Sexo', 'Motivo1', 'Motivo2'],
      dtype='object')

### Filter records with desired target group age (2-19)

In [202]:
data.shape

(17873, 17)

In [203]:

data = data[data['Idade'] != '#!VALUE!']
data = data[data['Idade'] != '#VALUE!']

In [204]:
data.shape

(17753, 17)

In [205]:
data['Idade'] = data['Idade'].astype(float)


In [206]:
data = data[data['Idade'].between(2,19)]
data.shape

(12375, 17)

In [207]:
data['Idade'].value_counts

<bound method IndexOpsMixin.value_counts of 4         9.60
5         4.40
6        12.89
7         5.89
10        6.24
         ...  
17862    12.30
17863     8.30
17865     8.67
17867     2.14
17872    11.38
Name: Idade, Length: 12375, dtype: float64>

### Handle Inconsistent values

In [208]:
# Patologia
print("(Before)", data['Patologia'].value_counts())
map = {
    'anormal': 'Anormal',
    'Normais': 'Normal'
}
data['Patologia'] = data['Patologia'].replace(map)
print("(After)", data['Patologia'].value_counts())



(Before) Patologia
Normal     7764
Anormal    4310
anormal       1
Normais       1
Name: count, dtype: int64
(After) Patologia
Normal     7765
Anormal    4311
Name: count, dtype: int64


In [209]:
# Sexo
print("(Before)",data['Sexo'].value_counts())
data['Sexo'] = data['Sexo'].str.strip().str.capitalize()
map = {
    'M': 'Masculino',
    'F': 'Feminino'
}
data['Sexo'] = data['Sexo'].replace(map)
print("(After)", data['Sexo'].value_counts())

(Before) Sexo
M                6641
F                4882
Masculino         463
Feminino          178
Indeterminado     146
masculino          62
Name: count, dtype: int64
(After) Sexo
Masculino        7166
Feminino         5060
Indeterminado     146
Name: count, dtype: int64


In [210]:
# Pulsos
print("(Before)",data['Pulsos'].value_counts())
data['Pulsos'] = data['Pulsos'].str.strip().str.capitalize()
map = {
    'Normais': 'Normal',
    'NORMAIS': 'Normal',
    'Diminuídos': 'Femorais diminuidos',
}
data['Pulsos'] = data['Pulsos'].replace(map)
print("(After)", data['Pulsos'].value_counts())


(Before) Pulsos
Normais                12010
Outro                     26
Amplos                    17
Femorais diminuidos        9
Diminuídos                 7
NORMAIS                    1
Name: count, dtype: int64
(After) Pulsos
Normal                 12011
Outro                     26
Amplos                    17
Femorais diminuidos       16
Name: count, dtype: int64


In [211]:
# B2
print("(Before)",data['B2'].value_counts())
data['B2'] = data['B2'].str.strip().str.capitalize()
map = {
    'Única': 'Unico',
    'Desdob fixo': 'Split fixo'
}
data['B2'] = data['B2'].replace(map)
print("(After)", data['B2'].value_counts())

(Before) B2
Normal           11720
Hiperfonética      136
Desdob fixo        115
Outro               79
Única               28
Name: count, dtype: int64
(After) B2
Normal           11720
Hiperfonética      136
Split fixo         115
Outro               79
Unico               28
Name: count, dtype: int64


In [212]:
# Sopro
print("(Before)",data['Sopro'].value_counts())
data['Sopro'] = data['Sopro'].str.strip().str.capitalize()
map = {
    'ausente': 'Ausente',
    'sistólico': 'Sistólico',
    'contínuo': 'Contínuo',
    'diastólico': 'Diastólico'
}
data['Sopro'] = data['Sopro'].replace(map)
print("(After)", data['Sopro'].value_counts())

(Before) Sopro
ausente                   8254
Sistólico                 3019
sistólico                  771
contínuo                    16
Contínuo                    13
diastólico                   8
Sistolico e diastólico       2
Name: count, dtype: int64
(After) Sopro
Ausente                   8254
Sistólico                 3790
Contínuo                    29
Diastólico                   8
Sistolico e diastólico       2
Name: count, dtype: int64


In [213]:
# Motivo1
data['Motivo1'].value_counts()
print("(Before)",data['Motivo1'].value_counts())
map = {
    '5 - Parecer cardiológico' : 'Triagem Cardiológica',
    '6 - Suspeita de cardiopatia': 'Possivel cardiopatia',
    '2 - Check-up' : 'Checkup de rotina',
    '1 - Cardiopatia já estabelecida': 'Cardiopatia',
    '7 - Outro' : 'Outro'
}
data['Motivo1'] = data['Motivo1'].replace(map)
print("(After)", data['Motivo1'].value_counts())

(Before) Motivo1
5 - Parecer cardiológico           6470
6 - Suspeita de cardiopatia        3911
2 - Check-up                        788
1 - Cardiopatia já estabelecida     784
7 - Outro                           314
Name: count, dtype: int64
(After) Motivo1
Triagem Cardiológica    6470
Possivel cardiopatia    3911
Checkup de rotina        788
Cardiopatia              784
Outro                    314
Name: count, dtype: int64


In [216]:
# Motivo2
data['Motivo2'].value_counts()
print("(Before)",data['Motivo2'].value_counts())
map = {
    '5 - Cirurgia': 'Cirurgia',
    '6 - Sopro' : 'Presença de Sopro',
    '5 - Atividade física' : 'Atividade física',
    'Outro' : 'Outros',
    '1 - Cardiopatia congenica' : 'Cardiopatia congenica',
    '6 - Dor precordial' : 'Outros',
    '6 - Palpitação/taquicardia/arritmia' : 'Outros',
    '6 - HAS/dislipidemia/obesidade' : 'Fatores de risco',
    '6 - Dispnéia' : 'Outros',
    '1 - Cardiopatia adquirida' :'Outros',
    '6 - Cianose' : 'Outros',
    '6 - Cardiopatia na familia' : 'Fatores de risco',
    '6 - Cansaço' : 'Outros',
    '6 - Alterações de pulso/perfusão' : 'Outros',
    '6 - Cianose e dispnéia' : 'Outros',
    '5 - Uso de cisaprida' : 'Outros'

}
data['Motivo2'] = data['Motivo2'].replace(map)
print("(After)", data['Motivo2'].value_counts())


(Before) Motivo2
5 - Cirurgia                           3351
6 - Sopro                              1718
5 - Atividade física                   1069
Outro                                   719
1 - Cardiopatia congenica               639
6 - Dor precordial                      615
6 - Palpitação/taquicardia/arritmia     452
6 - HAS/dislipidemia/obesidade          395
6 - Dispnéia                            240
1 - Cardiopatia adquirida               112
6 - Cianose                              51
6 - Cardiopatia na familia               31
6 - Cansaço                              16
6 - Alterações de pulso/perfusão          2
6 - Cianose e dispnéia                    2
5 - Uso de cisaprida                      1
Name: count, dtype: int64
(After) Motivo2
Cirurgia                 3351
Outros                   2210
Presença de Sopro        1718
Atividade física         1069
Cardiopatia congenica     639
Fatores de risco          426
Name: count, dtype: int64


### Handling incorect/impossible/irrelevant values

TODO:
1) Ajustar SBP e DBP para valores de referencia (Remover valores que não estejam dentro dos valores de referencia, não esquecer de adicionar)
2) Verificar valores impossiveis de peso e altura 
3) Remover valores Pulso = Outro (Justificar)
4) Remover valores B2 = Outro (Justificar)
5) Remover valores Sopro = Sistolico e diastólico (Justicar)
6) Corrigir Frequencia cardiacas (valores que não fazem sentido, e secalhar definir intervalos de frequencia)
7) Remover valores HDA1,HDA2 = outros (Justificar)
8) Remover HDA2= Assintomático (Justificar)
9) Remover Sexo = Indeterminado (Justificar)

### TODO

1) Recalcular BMI (remover os passos do valor de refrencia)
2) Recalcular Result SBP-PPA  

> Usar o document.pdf como referência

## Data Preprocessing